# Phase 14 — Preference Dataset Construction
## CodeMentor-LLM
Building preference dataset for DPO training.
Chosen = SFT model response
Rejected = Base model response

## Goal
- Generate chosen/rejected pairs from SFT and base model
- Format as DPO preference dataset
- Push to HuggingFace Hub

# libraries

In [ ]:
!pip uninstall numpy -y
!pip install numpy==1.26.4
!pip install -q transformers==4.49.0 peft==0.14.0 bitsandbytes==0.45.3 accelerate==1.5.1 datasets==3.3.2

# Login to HuggingFace

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Logged in successfully")

# Load model and tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

model_id = "meta-llama/Llama-3.2-3B-Instruct"
adapter_id = "Abdulmoiz123/codementor-llm-sft"

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

# Load SFT adapter
print("Loading SFT adapter...")
sft_model = PeftModel.from_pretrained(base_model, adapter_id)
print("Model loaded successfully")

#  Load dataset and define functions

In [ ]:
from datasets import load_dataset

# Load train split
print("Loading dataset...")
dataset = load_dataset("Abdulmoiz123/codementor-llm-splits")
train_dataset = dataset["train"].select(range(200))
print(f"Samples loaded: {len(train_dataset)}")

In [ ]:
SYSTEM_PROMPT = "You are a helpful coding assistant. Answer coding questions clearly and concisely with working code examples."

def extract_instruction(text):
    if "<|start_header_id|>user<|end_header_id|>" in text:
        instruction = text.split("<|start_header_id|>user<|end_header_id|>")[-1]
        instruction = instruction.split("<|eot_id|>")[0].strip()
        return instruction
    return ""

def generate_response(model, tokenizer, instruction, max_new_tokens=128):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3,
        )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )
    return response

print("Functions defined successfully")

#  Generate preference pairs

In [ ]:
import json

print("Generating preference pairs...")

preference_pairs = []
for i, sample in enumerate(train_dataset):
    instruction = extract_instruction(sample["text"])

    if not instruction:
        continue

    # Generate SFT response (chosen)
    sft_model.enable_adapter_layers()
    chosen = generate_response(sft_model, tokenizer, instruction)

    # Generate base response (rejected)
    sft_model.disable_adapter_layers()
    rejected = generate_response(sft_model, tokenizer, instruction)

    # Format as DPO pair
    preference_pairs.append({
        "prompt": instruction,
        "chosen": chosen,
        "rejected": rejected
    })

    if (i+1) % 100 == 0:
        print(f"Progress: {i+1}/200")

print(f"\nTotal preference pairs: {len(preference_pairs)}")

#  Review 10 samples manually

In [ ]:
print("MANUAL REVIEW — 10 SAMPLES")

for i in range(10):
    print(f"\nSample {i+1}")
    print(f"PROMPT  : {preference_pairs[i]['prompt']}")
    print(f"\nCHOSEN  : {preference_pairs[i]['chosen']}")
    print(f"\nREJECTED: {preference_pairs[i]['rejected']}")
    print("=" * 60)

# Push preference dataset to HuggingFace Hub

In [ ]:
from datasets import Dataset

# Convert to HuggingFace Dataset
preference_dataset = Dataset.from_list(preference_pairs)

print(f"Preference dataset size: {len(preference_dataset)}")
print(f"Columns: {preference_dataset.column_names}")

# Push to HF Hub
print("\nPushing to HuggingFace Hub...")
preference_dataset.push_to_hub("Abdulmoiz123/codementor-llm-preference")
print("Preference dataset pushed successfully")

# Save Dataset Locally

In [ ]:
import json

output_file = "preference_pairs.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for pair in preference_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

print(f"Saved {len(preference_pairs)} pairs to {output_file}")